# Ensemble de semillas — promedio de probabilidades y submit a Kaggle

Para un experimento dado:
1. Lee los `prediccion_<experiment_name>-<semilla>.txt` de cada una de sus semillas (generados por `predict_and_save()` en el workflow original).
2. Promedia la probabilidad (`prob`) por `numero_de_cliente`.
3. Graba el promedio en `prediccion_<experiment_name>-avg.txt`.
4. Sube a Kaggle con los mismos cortes (`PARAM$kaggle$cortes`) y el mismo estilo de archivo/comando que `kaggle_submit()` — el nombre de archivo usa `avg` donde antes iba la semilla: `KA<experimento>_<experiment_name>-avg_<corte>.csv`.

## 1) Librerías

In [1]:
suppressMessages({
  require("data.table")
})


## 2) Configuración

- `carpeta_experimento`: misma carpeta donde están los `prediccion_*.txt`, `results_summary_*.txt`, y donde se crea la subcarpeta `kaggle/`.
- `experimento`: número usado en el nombre de archivo (`KA9105_...`).
- `experiment_name`: el experimento puntual a ensamblar (ej. `"lags_deltas"`).
- `semillas`: las semillas de ese experimento a promediar (las 10 generadas con `PARAM$semilla_primigenia` / `PARAM$qsemillas` en el notebook original).

In [ ]:
PARAM <- list()
PARAM$carpeta_experimento <- "/content/buckets/b1/exp/WF9200"   # <-- AJUSTAR
setwd(PARAM$carpeta_experimento)

PARAM$experimento <- 9200                                        # <-- AJUSTAR si hace falta

# Experimento a ensamblar: 
# - Catastrophe Analysis -> ML; 
# - Data drifting -> estandarizar; 
# - FE intra-mes -> baseline; 
# - FE historica -> baseline (lag 1 y 2, delta 1 y 2); 
# - meses de pandemia no excluidos.
PARAM$experiment_name <- "CA-MachineLearning_DR-estandarizar_FEintra-SI_roll0"

# Las 10 semillas de ese experimento (las que genera
# PARAM$semilla_primigenia / PARAM$qsemillas en el notebook original)
PARAM$semillas <- c(468889, 567793, 347671, 678607, 702787, 117809, 925387, 382961, 744559, 777199)      # <-- AJUSTAR

PARAM$kaggle <- list()
PARAM$kaggle$competencia <- "utn-2026-virtual-jr"
PARAM$kaggle$cortes <- seq(1800, 2400, by = 100)
PARAM$kaggle$delay_segundos <- 30


## 3) Leer y promediar las predicciones de las semillas

Cada `prediccion_<experiment_name>-<semilla>.txt` tiene columnas `numero_de_cliente` y `prob` (separadas por tab), tal cual las graba `predict_and_save()`.

In [3]:
leer_y_promediar_predicciones <- function(experiment_name, semillas) {

  tablas <- list()

  for (semilla in semillas) {
    archivo <- sprintf("prediccion_%s-%d.txt", experiment_name, semilla)

    if (!file.exists(archivo)) {
      stop(sprintf("No encontre el archivo: %s", archivo))
    }

    tb <- fread(archivo, sep = "\t")
    tb[, semilla := semilla]
    tablas[[length(tablas) + 1]] <- tb
  }

  tb_todas <- rbindlist(tablas)

  # promedio de probabilidad por cliente, a traves de las semillas
  tb_avg <- tb_todas[, .(prob = mean(prob)), by = numero_de_cliente]

  return(tb_avg)
}


## 4) Ejecutar el promedio y grabar el archivo
Genera `prediccion_<experiment_name>-avg.txt`, con el mismo formato (`numero_de_cliente`, `prob`, separado por tab) que los archivos originales por semilla.

In [4]:
tb_prediccion_avg <- leer_y_promediar_predicciones(PARAM$experiment_name, PARAM$semillas)

archivo_avg <- sprintf("prediccion_%s-avg.txt", PARAM$experiment_name)
fwrite(tb_prediccion_avg, file = archivo_avg, sep = "\t")

cat(sprintf("Promedio de %d semillas para '%s' guardado en: %s (%d clientes)\n",
  length(PARAM$semillas), PARAM$experiment_name, archivo_avg, nrow(tb_prediccion_avg)))


Promedio de 10 semillas para 'CA-MachineLearning_DR-estandarizar_FEintra-SI_roll0' guardado en: prediccion_CA-MachineLearning_DR-estandarizar_FEintra-SI_roll0-avg.txt (33080 clientes)


## 5) AUC promedio (para el comentario del submit)

Busca en `results_summary_<experiment_name>.txt` el `best_auc` de cada semilla promediada, y calcula el promedio. Si falta alguna, lo avisa pero sigue igual (el comentario del submit va a aclarar cuántas se encontraron).

In [5]:
buscar_auc_promedio <- function(experiment_name, semillas) {
  archivo <- sprintf("results_summary_%s.txt", experiment_name)

  if (!file.exists(archivo)) {
    return(list(auc_promedio = NA, encontradas = 0, total = length(semillas)))
  }

  tb <- fread(archivo)
  filas <- tb[seed %in% semillas]

  if (nrow(filas) == 0) {
    return(list(auc_promedio = NA, encontradas = 0, total = length(semillas)))
  }

  # si alguna semilla aparece mas de una vez, me quedo con la ultima corrida
  filas <- filas[, .SD[.N], by = seed]

  list(
    auc_promedio = mean(filas$best_auc),
    encontradas = nrow(filas),
    total = length(semillas)
  )
}

auc_info <- buscar_auc_promedio(PARAM$experiment_name, PARAM$semillas)
auc_info


$auc_promedio
[1] 0.9215999

$encontradas
[1] 10

$total
[1] 10

## 6) Submit a Kaggle — mismo estilo que `kaggle_submit()` original

Mismos cortes (`PARAM$kaggle$cortes`), mismo formato de archivo/comando, sin `shQuote` en `-f`, ruta relativa `./kaggle/...`. El nombre de archivo usa `avg` en lugar de la semilla.

In [6]:
kaggle_submit_avg <- function(experiment_name, tb_prediccion, semillas, auc_info) {

  # ordeno por probabilidad descendente, igual que kaggle_submit()
  setorder(tb_prediccion, -prob)

  dir.create("kaggle", showWarnings = FALSE)

  semillas_str <- paste(semillas, collapse = ",")
  auc_str <- if (is.na(auc_info$auc_promedio)) {
    "N/D"
  } else {
    sprintf("%.10f (de %d/%d semillas encontradas en results_summary)",
      auc_info$auc_promedio, auc_info$encontradas, auc_info$total)
  }

  for (envios in PARAM$kaggle$cortes) {

    tb_prediccion[, Predicted := 0L] # seteo inicial a 0
    tb_prediccion[1:envios, Predicted := 1L] # marco los primeros

    archivo_kaggle <- sprintf(
        "./kaggle/KA%d_%s-avg_%d.csv", # KA9105_lags_deltas-avg_1800.csv
        PARAM$experimento,
        experiment_name,
        envios
    )

    # grabo el archivo
    fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
      file = archivo_kaggle,
      sep = ","
    )

    # subida a Kaggle, armo la linea de comando
    kaggle <- file.path(Sys.getenv("HOME"), ".venv", "bin", "kaggle")
    comando <- paste(shQuote(kaggle), "competitions submit")
    competencia <- paste("-c", PARAM$kaggle$competencia)
    arch <- paste("-f", archivo_kaggle)

    mensaje <- sprintf(
        "-m 'experimento=%s-avg envios=%d semillas=[%s] \n\nEnsemble: promedio de %d semillas.\nAUC promedio = %s'",
        experiment_name,
        envios,
        semillas_str,
        length(semillas),
        auc_str
    )

    linea <- paste(comando, competencia, arch, mensaje)

    salida <- system(linea, intern = TRUE) # el submit a Kaggle
    cat(salida, "\n")
    flush.console()
    Sys.sleep(PARAM$kaggle$delay_segundos)
  }
}


## 7) Ejecutar el submit
Esta es la única celda que efectivamente sube archivos a Kaggle (un submit por cada corte en `PARAM$kaggle$cortes`, esperando `PARAM$kaggle$delay_segundos` entre cada uno).

In [ ]:
kaggle_submit_avg(PARAM$experiment_name, tb_prediccion_avg, PARAM$semillas, auc_info)

92 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
91 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
90 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
89 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
88 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
87 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
86 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
